# MNIST

**Objetivo:** entrenar una red neuronal simple, que reconozca dígitos escritos a mano del dataset (MNIST)

**Framework utilizado: Tensorflow / Keras**. La elección se debe a que su API Sequential permite estructurar redes densas de forma rápida y directa. Ideal para comparar activaciones y optimizadores en pocas líneas de código.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix

# SEED -> Garantiza la reproducibilidad de los resultados
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## 1. Carga y exploración de los datos

MNINST trae 70.000 imágenes de 28x28 píxeles en escalses de grises. Uso (60.000) para entrenar, 10.000 para test). Cada imagen tiene una etiqueta del 0 al 9

In [ ]:
# Descarga MNIST desde Keras
(X_train_img, y_train), (X_test_img, y_test) = keras.datasets.mnist.load_data()

# Mostramos cuántas imágenes hay y de que tamaño son
print("Train: ", X_train_img.shape)
print("Test: ", X_test_img.shape)

# Muestra 10 ejemplos
n_mostrar = 10
plt.figure(figsize=(10,2))
for i in range(n_mostrar):
    plt.subplot(1, n_mostrar, i+1)
    plt.imshow(X_train_img[i], cmap="gray")
    plt.title(str(y_train[i]))
    plt.axis("off")

plt.show()

## Preprocesamiento: normalizar y aplanar

- **Normalizar:** los píxeles van de 0 a 255. Se divide por 255 para dejarlos en 0 y 1, así la red aprende más rápido.
- **Aplanar:** la red densa no entiende imágenes 2D, así que cada imagen de 28x28 la convertimos en un vector de 784 números.

In [ ]:
INPUT_DIM = 28 * 28
X_train = (X_train_img.astype("float32") / 255.0).reshape(-1, INPUT_DIM)
X_test = (X_test_img.astype("float32") / 255).reshape(-1, INPUT_DIM)

print(f"Train aplanado: {X_train.shape}")
print(f"Valor mínimo: {X_train.min()} | máximo: {X_train.max()}")

## 3. Arquitectura: el cerebro

Red con **1 sola capa oculta**
- **Entrada:** 784 neuronas (una por píxel)
- **Capa Oculta:** 128 neuronas con activación ReLU o Sigmoid (para comparar)
- **Salida:** 10 neuronas con softmax (una probabilidad por cada dígito 0-9)

In [ ]:
# Función que arma la misma red cambiando solo activación y optimizador
def crear_modelo(activacion, optimizador):
    # LR igual para comparar de forma justa
    lr = 0.001
    if optimizador == "adam":
        opt = keras.optimizers.Adam(learning_rate=lr)
    else:
        opt = keras.optimizers.SGD(learning_rate=lr)

    modelo = keras.Sequential([
        keras.layers.Input(shape=(784,), name="entrada"),
        keras.layers.Dense(128, activation=activacion, name="oculta"),
        keras.layers.Dense(10, activation="softmax", name="salida")
    ])

    modelo.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return modelo

# Mostrar el resumen con ReLU + Adam como ejemplo
modelo_demo = crear_modelo(activacion="relu", optimizador="adam")
modelo_demo.summary()